# 🏸 AI-Based Badminton Shot Prediction System
### Decision Tree Classifier for Shot Recommendation
**Course Project | AI/ML | BYOP Submission**

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
import joblib
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## 2. Load & Explore the Dataset

In [ ]:
df = pd.read_csv('data/badminton_dataset.csv')
print(f'Dataset Shape: {df.shape}')
df.head(10)

In [ ]:
print('--- Dataset Info ---')
df.info()
print('\n--- Statistical Summary ---')
df.describe()

In [ ]:
print('--- Missing Values ---')
print(df.isnull().sum())
print('\n--- Class Distribution ---')
print(df['Recommended_Shot'].value_counts())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Badminton Shot Prediction - EDA', fontsize=16, fontweight='bold')

# Shot distribution
shot_counts = df['Recommended_Shot'].value_counts()
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
axes[0].pie(shot_counts, labels=shot_counts.index, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Shot Type Distribution')

# Shuttle Difficulty vs Shot
pd.crosstab(df['Shuttle_Difficulty'], df['Recommended_Shot']).plot(
    kind='bar', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Shuttle Difficulty vs Shot Type')
axes[1].set_xlabel('Shuttle Difficulty')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Shot')

# Score Situation vs Shot
pd.crosstab(df['Score_Situation'], df['Recommended_Shot']).plot(
    kind='bar', ax=axes[2], color=colors, edgecolor='white')
axes[2].set_title('Score Situation vs Shot Type')
axes[2].set_xlabel('Score Situation')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(title='Shot')

plt.tight_layout()
plt.savefig('eda_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA plot saved.')

In [ ]:
# Scatter plot - Player positions coloured by shot
fig, ax = plt.subplots(figsize=(8, 10))
colors_map = {'Smash': '#FF6B6B', 'Drop': '#4ECDC4', 'Clear': '#45B7D1'}
for shot, color in colors_map.items():
    subset = df[df['Recommended_Shot'] == shot]
    ax.scatter(subset['Player_X'], subset['Player_Y'], c=color, label=shot, alpha=0.6, s=60)

# Draw court outline
court = plt.Polygon([[0,0],[10,0],[10,13.4],[0,13.4]], fill=False, edgecolor='black', linewidth=2)
ax.add_patch(court)
ax.axhline(y=6.7, color='gray', linestyle='--', linewidth=1, label='Net')
ax.set_xlim(-0.5, 10.5)
ax.set_ylim(-0.5, 14)
ax.set_xlabel('Court Width (X)')
ax.set_ylabel('Court Length (Y)')
ax.set_title('Player Position vs Shot Type')
ax.legend()
plt.tight_layout()
plt.savefig('court_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Data Preprocessing

In [ ]:
# Label Encoding for categorical features
le_score = LabelEncoder()
le_shuttle = LabelEncoder()
le_shot = LabelEncoder()

df_encoded = df.copy()
df_encoded['Score_Situation'] = le_score.fit_transform(df['Score_Situation'])
df_encoded['Shuttle_Difficulty'] = le_shuttle.fit_transform(df['Shuttle_Difficulty'])
df_encoded['Recommended_Shot'] = le_shot.fit_transform(df['Recommended_Shot'])

print('Score Situation encoding:', dict(zip(le_score.classes_, le_score.transform(le_score.classes_))))
print('Shuttle Difficulty encoding:', dict(zip(le_shuttle.classes_, le_shuttle.transform(le_shuttle.classes_))))
print('Shot encoding:', dict(zip(le_shot.classes_, le_shot.transform(le_shot.classes_))))

df_encoded.head()

In [ ]:
# Split features and target
X = df_encoded.drop('Recommended_Shot', axis=1)
y = df_encoded['Recommended_Shot']

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Testing set:  {X_test.shape[0]} samples')

## 5. Model Training - Decision Tree Classifier

In [ ]:
# Train Decision Tree
dt_model = DecisionTreeClassifier(
    max_depth=6,
    min_samples_split=5,
    min_samples_leaf=3,
    random_state=42
)

dt_model.fit(X_train, y_train)
print('Decision Tree model trained successfully!')
print(f'Tree Depth: {dt_model.get_depth()}')
print(f'Number of Leaves: {dt_model.get_n_leaves()}')

In [ ]:
# Predictions
y_pred = dt_model.predict(X_test)

# Accuracy
train_acc = accuracy_score(y_train, dt_model.predict(X_train))
test_acc = accuracy_score(y_test, y_pred)

print(f'Training Accuracy : {train_acc*100:.2f}%')
print(f'Testing Accuracy  : {test_acc*100:.2f}%')

## 6. Evaluation Metrics

In [ ]:
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=le_shot.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le_shot.classes_)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix - Badminton Shot Prediction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance
feature_names = X.columns.tolist()
importances = dt_model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar([feature_names[i] for i in sorted_idx], importances[sorted_idx],
               color=['#FF6B6B','#4ECDC4','#45B7D1','#96CEB4','#FFEAA7','#DDA0DD'])
ax.set_title('Feature Importance - Decision Tree', fontsize=13, fontweight='bold')
ax.set_ylabel('Importance Score')
ax.set_xlabel('Features')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

for i in sorted_idx:
    print(f'{feature_names[i]:25s}: {importances[i]:.4f}')

In [ ]:
# Visualize the Decision Tree
fig, ax = plt.subplots(figsize=(22, 10))
plot_tree(
    dt_model,
    feature_names=feature_names,
    class_names=le_shot.classes_,
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax
)
ax.set_title('Decision Tree Visualization', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('decision_tree.png', dpi=120, bbox_inches='tight')
plt.show()
print('Tree visualization saved.')

## 7. Prediction Examples

In [ ]:
def predict_shot(player_x, player_y, opponent_x, opponent_y,
                  score_situation, shuttle_difficulty):
    """
    Predict the best badminton shot given court conditions.
    """
    ss_enc = le_score.transform([score_situation])[0]
    sd_enc = le_shuttle.transform([shuttle_difficulty])[0]

    input_data = pd.DataFrame([{
        'Player_X': player_x,
        'Player_Y': player_y,
        'Opponent_X': opponent_x,
        'Opponent_Y': opponent_y,
        'Score_Situation': ss_enc,
        'Shuttle_Difficulty': sd_enc
    }])

    pred_enc = dt_model.predict(input_data)[0]
    pred_proba = dt_model.predict_proba(input_data)[0]
    shot = le_shot.inverse_transform([pred_enc])[0]

    proba_dict = dict(zip(le_shot.classes_, pred_proba))
    return shot, proba_dict

# Example 1: Player at back, easy shuttle, opponent near net -> Smash expected
shot, proba = predict_shot(5.0, 12.0, 5.0, 1.0, 'Pressure', 'Easy')
print(f'Example 1 -> Predicted Shot: {shot}')
print(f'  Probabilities: {proba}\n')

# Example 2: Player near net, medium shuttle -> Drop expected
shot, proba = predict_shot(4.5, 3.0, 5.0, 8.0, 'Mid', 'Medium')
print(f'Example 2 -> Predicted Shot: {shot}')
print(f'  Probabilities: {proba}\n')

# Example 3: Hard shuttle, defensive position -> Clear expected
shot, proba = predict_shot(3.0, 12.5, 6.0, 7.0, 'Pressure', 'Hard')
print(f'Example 3 -> Predicted Shot: {shot}')
print(f'  Probabilities: {proba}')

## 8. Save the Model

In [ ]:
import os
os.makedirs('model', exist_ok=True)
joblib.dump(dt_model, 'model/badminton_model.pkl')
joblib.dump(le_score, 'model/le_score.pkl')
joblib.dump(le_shuttle, 'model/le_shuttle.pkl')
joblib.dump(le_shot, 'model/le_shot.pkl')
print('Model and encoders saved to model/ directory!')

## 9. Summary

| Metric | Value |
|--------|-------|
| Algorithm | Decision Tree Classifier |
| Dataset Size | 300 samples |
| Train/Test Split | 80% / 20% |
| Target Classes | Smash, Drop, Clear |
| Max Tree Depth | 6 |

**Why Decision Tree?**
- Interpretable: Can explain WHY a shot was recommended
- Works well with both numerical and categorical features
- No need for feature scaling
- Fast inference — critical for real-time use